# 🇱🇰 Sinhala Script Robustness — Full Visualization & Analysis Notebook
### Extrinsic Evaluation Visualizations & Multi-Task Robustness Analysis

---

## 🎯 What is this Project About?

People write the Sinhala language in two main ways:
1. **Unicode (Native Script):** Writing using real Sinhala letters (e.g., `ආයුබෝවන්`).
2. **Romanized (Singlish / Transliterated):** Writing Sinhala words using English letters (e.g., `ayubowan`).

**Core Question:** Can popular Artificial Intelligence (AI) Language Models understand both scripts equally well, or do they fail when switching between native Sinhala letters and Singlish?

---

## 📚 Datasets & Tasks Evaluated

We tested the models on three distinct natural language tasks:
1. **Sinhala MMLU (6,879 questions):** Multiple-choice exam questions covering science, history, law, and general knowledge translated into Sinhala.
2. **SOLD (2,500 comments):** Sinhala Offensive Language Dataset. Deciding if a comment is **Offensive (OFF)** or **Not Offensive (NOT)**.
3. **Global PIQA (100 questions):** Physical Commonsense Reasoning translated into Sinhala (e.g., everyday physical facts and solutions).

---

## 🤖 Models Tested (10 AI Models)

| Model Name | Size (Parameters) | Description |
|:---|:---|:---|
| **TinyLlama-1.1B** | 1.1 Billion | Very lightweight small model |
| **LaMini-GPT-1.5B** | 1.5 Billion | Small instruction-tuned model |
| **SmolLM3-3B** | 3.0 Billion | Modern lightweight small language model |
| **StableLM-Zephyr-3B** | 3.0 Billion | Compact instruction-following model |
| **Qwen-3.5-4B** | 4.0 Billion | Medium-sized multilingual model from Alibaba |
| **Qwen2-7B-Instruct** | 7.0 Billion | Strong 7B open-source model |
| **Zephyr-7B-Beta** | 7.0 Billion | Popular alignment-tuned model |
| **Hormoz-8B** | 8.0 Billion | Multilingual model trained on regional languages |
| **Llama-3.1-8B-Instruct** | 8.0 Billion | State-of-the-art 8B open model from Meta |
| **Qwen-3.5-9B** | 9.0 Billion | Advanced 9B multilingual model from Alibaba |

---


## 🛠️ Step 1: Environment Setup & Data Loading

In the cell below, we load all necessary Python libraries (Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn) and verify that our dataset records are loaded properly.

In [ ]:
import os, warnings, numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
warnings.filterwarnings('ignore')

_nb_dir = os.path.dirname(os.path.abspath('__file__'))
BASE    = os.path.normpath(os.path.join(_nb_dir, '..', 'extrinsic_evaluation'))
OUT_DIR = os.path.normpath(os.path.join(_nb_dir, 'figures'))
print('All libraries loaded successfully!')

In [ ]:
# Load evaluation summary for all 10 models
MODELS = {
    'TinyLlama-1.1B':  'TinyLlama-1.1B-Chat-v1.0',
    'LaMini-GPT-1.5B': 'LaMini-GPT-1.5B',
    'SmolLM3-3B':      'SmolLM3-3B',
    'StableLM-3B':     'stablelm-zephyr-3b',
    'Qwen2-7B':        'Qwen2-7B-Instruct',
    'Zephyr-7B':       'zephyr-7b-beta',
    'Hormoz-8B':       'Hormoz-8B',
    'Llama-3.1-8B':    'Llama-3.1-8B-Instruct',
    'Qwen3.5-4B':      'Qwen-3.5-4B',
    'Qwen3.5-9B':      'Qwen-3.5-9B',
}

records = []
for m_name, folder in MODELS.items():
    files = [f for f in os.listdir(os.path.join(BASE, folder)) if 'summary' in f]
    if not files: continue
    df = pd.read_csv(os.path.join(BASE, folder, files[0]))
    for _, row in df.iterrows():
        if pd.notna(row.get('Task')):
            records.append({'Model': m_name, 'Task': row['Task'],
                'Unicode_Acc': row['Unicode Acc (%)'], 'Romanized_Acc': row['Romanized Acc (%)'],
                'Gap': row['Gap (pp)'], 'N': row['N'],
                'Unicode_Invalid': row['Unicode Invalid (%)'],
                'Romanized_Invalid': row['Romanized Invalid (%)'],
            })

summary_df = pd.DataFrame(records)
print(summary_df.to_string())

---


## 📊 Figure 1: Accuracy Comparison (Native Unicode vs Romanized Singlish)

### 📌 Simple Summary:
This chart shows how accurately each model answered questions when given **native Sinhala script** compared to when given **Romanized Singlish (English letters)**.

### 🔍 How to Read This Chart:
- **Blue Bar:** Accuracy on native Sinhala text (`Unicode`).
- **Pink Bar:** Accuracy on English-letter Sinhala text (`Romanized`).
- **Taller bar = Better performance.**
- The 3 sub-graphs represent our 3 datasets: **Sinhala MMLU**, **SOLD**, and **Global PIQA**.

### 💡 What the Data Tells Us:
1. **Big models prefer native Sinhala on complex tasks:** On Sinhala MMLU, models like `Qwen-3.5-9B` and `Llama-3.1-8B` have much taller blue bars than pink bars. They understand real Sinhala script much better than phonetic English spelling.
2. **Simple tasks show less difference:** On the SOLD dataset (offensive comment detection), blue and pink bars are almost equal for most models (~60%).
3. **Small models struggle everywhere:** `TinyLlama` and `LaMini` have tiny bars (<25%), which is no better than guessing randomly.

In [ ]:
# Display: Figure 1: Unicode vs Romanized Accuracy — Grouped Bar Chart
from IPython.display import Image, display
display(Image(r"figures/fig01_grouped_bar_accuracy.png", width=1100))

---


## 🌡️ Figure 2: The Script Performance Gap Heatmap

### 📌 Simple Summary:
This heatmap calculates the exact difference: **(Unicode Accuracy − Romanized Accuracy)** in percentage points.

### 🔍 How to Read the Colors:
- 🟢 **Green (Positive Number):** The model did **better on native Sinhala script**.
- 🔴 **Red (Negative Number):** The model did **better on Romanized Singlish**.
- 🟡 **Yellow (Close to Zero):** The model treated **both scripts equally** (This means it is **Script-Robust**).

### 💡 What the Data Tells Us:
1. **Qwen-3.5-9B (+15.47 on MMLU):** Shows a bright green box. It gets a 15% boost when using native Sinhala letters because its tokenizer and training included real Sinhala data.
2. **SmolLM3-3B (+0.76 on MMLU, +1.64 on SOLD):** Shows soft yellow boxes. It performs almost identically regardless of script, showing high script consistency.
3. **LaMini-GPT-1.5B (-18.08 on SOLD, -17.87 on MMLU):** Shows bright red boxes. It fails badly on native Sinhala script and only gets answers right when written in English letters.

In [ ]:
# Display: Figure 2: Performance Gap Heatmap (Unicode - Romanized)
from IPython.display import Image, display
display(Image(r"figures/fig02_gap_heatmap.png", width=1100))

---


## ❌ Figure 3: Invalid / Broken Response Rates

### 📌 Simple Summary:
When an AI model is asked a multiple-choice question, it should reply with a valid option (like 'A', 'B', 'C', 'NOT', or 'OFF'). If the model outputs nonsense, repeats words endlessly, or leaves it blank, it is counted as an **Invalid Response**.

### 🔍 How to Read This Chart:
- **Left side:** Invalid rate for native Sinhala script (`Unicode`).
- **Right side:** Invalid rate for Romanized script (`Romanized`).
- **0% (White/Pale):** Perfect! The model followed instructions and gave valid answers.
- **High % (Dark Red):** Failure! The model was confused by the text and could not generate a proper answer.

### 💡 What the Data Tells Us:
1. **LaMini-GPT-1.5B has a 92% to 95% failure rate on native Sinhala:** It cannot understand native Sinhala characters at all. Its tokenizer breaks Sinhala letters into unknown tokens (`<unk>`).
2. **TinyLlama-1.1B fails 95% of the time on Unicode MMLU:** It does not know Sinhala script.
3. **Modern 7B-9B models have 0.0% invalid rates:** Models like `Qwen2-7B`, `Llama-3.1-8B`, and `Qwen3.5-9B` always follow instructions properly in both scripts.

In [ ]:
# Display: Figure 3: Invalid Response Rate Heatmap
from IPython.display import Image, display
display(Image(r"figures/fig03_invalid_rate_heatmap.png", width=1100))

---


## 📈 Figure 4: ROC Curves (Offensive Language Detection)

### 📌 Simple Summary:
The **ROC Curve (Receiver Operating Characteristic)** tests how well an AI can separate **Offensive comments (OFF)** from **Clean comments (NOT)** without making false accusations.

### 🔍 How to Read This Chart:
- **X-axis (False Positive Rate):** How often clean comments were wrongly flagged as offensive.
- **Y-axis (True Positive Rate):** How often offensive comments were successfully caught.
- **Diagonal Dashed Line:** Random coin toss guessing (AUC = 0.50).
- **Top-Left Corner:** The closer the curve is to the top-left, the better the model (AUC closer to 1.00).

### 💡 What the Data Tells Us:
1. **Llama-3.1-8B and Qwen models achieve the highest AUC:** They detect hate speech and toxic comments with the fewest mistakes.
2. **Small models stay near the 0.50 diagonal line:** Small models cannot tell whether a Sinhala comment is rude or polite.

In [ ]:
# Display: Figure 4: ROC Curves — SOLD Task
from IPython.display import Image, display
display(Image(r"figures/fig04_roc_curves_sold.png", width=1100))

---


## 🎯 Figure 5: Precision-Recall Curves (Catching Toxic Comments)

### 📌 Simple Summary:
This graph looks specifically at how accurately models detect the **Offensive (OFF)** class.

### 🔍 How to Read This Chart:
- **Precision (Y-axis):** When the AI flags a comment as offensive, what percentage of the time is it actually offensive? (High = very few false alarms).
- **Recall (X-axis):** Out of all offensive comments in the dataset, how many did the AI successfully catch? (High = very few toxic comments missed).
- **AP (Average Precision):** Higher score = Better balance of catching toxic comments without false alarms.

### 💡 What the Data Tells Us:
1. In both native Sinhala and Singlish, **Llama-3.1-8B** achieves top detection ability.
2. Models that always say 'NOT offensive' drop to 0 on Recall because they miss every toxic comment.

In [ ]:
# Display: Figure 5: Precision-Recall Curves — SOLD Task
from IPython.display import Image, display
display(Image(r"figures/fig05_precision_recall_sold.png", width=1100))

---


## ⚖️ Figure 6: Detailed Classification Metrics (Precision, Recall, F1-Score)

### 📌 Simple Summary:
Accuracy alone can be misleading if a model just guesses the same answer every time. This chart breaks down performance into 3 key metrics:
1. **Precision:** Accuracy of positive flags.
2. **Recall:** Coverage of all real toxic cases.
3. **F1-Score:** The balanced average combining Precision and Recall.

### 🔍 How to Read This Chart:
- **Blue Bar = Native Sinhala (Unicode)** | **Pink Bar = Romanized Singlish**
- Score goes from **0.0 (Worst)** to **1.0 (Perfect)**.

### 💡 What the Data Tells Us:
1. **Llama-3.1-8B reaches an F1-score around 0.65:** It is currently the best model for moderation of Sinhala social media content.
2. **Script consistency:** For top models, the blue and pink F1 bars are very close in height, meaning they can detect hate speech in both native Sinhala script and Singlish equally well.

In [ ]:
# Display: Figure 6: Precision, Recall & F1-Score per Model (SOLD)
from IPython.display import Image, display
display(Image(r"figures/fig06_f1_precision_recall.png", width=1100))

---


## 🔢 Figure 7: Confusion Matrices (What Mistakes Do Models Make?)

### 📌 Simple Summary:
A confusion matrix shows the exact breakdown of correct answers vs mistakes.

### 🔍 How to Read Each 2x2 Box:
- **Top-Left Box (True NOT):** Clean comments correctly identified as clean. *(Good)*
- **Bottom-Right Box (True OFF):** Offensive comments correctly caught. *(Good)*
- **Top-Right Box (False Alarm):** Clean comment wrongly marked as offensive. *(Bad)*
- **Bottom-Left Box (Missed):** Toxic comment missed by the model. *(Bad)*
- **Row 1:** Results using Native Sinhala script.
- **Row 2:** Results using Romanized Singlish.

### 💡 What the Data Tells Us:
1. **Top models have dark diagonal boxes:** They correctly categorize both clean and offensive comments.
2. **Weak models put all numbers in the left column:** They simply predict 'NOT' for every single comment because they cannot read the text.

In [ ]:
# Display: Figure 7: Confusion Matrices — SOLD Task (All Models)
from IPython.display import Image, display
display(Image(r"figures/fig07_confusion_matrices_sold.png", width=1100))

---


## 🕸️ Figure 8: Radar / Spider Chart (Overall Capability Profile)

### 📌 Simple Summary:
This spider chart combines performance across all 3 tasks onto a single web. A **larger, wider shape** means a stronger, more capable AI model.

### 🔍 How to Read This Chart:
- **3 Corners = 3 Tasks:** MMLU (Knowledge), SOLD (Hate Speech), PIQA (Common Sense).
- **Left Web:** Native Sinhala Script (`Unicode`).
- **Right Web:** Romanized Script (`Romanized`).
- **Bigger colored area = Better all-around model.**

### 💡 What the Data Tells Us:
1. **Qwen-3.5-9B and Llama-3.1-8B cover the largest area:** They perform well across all three diverse tasks.
2. **Comparing Left vs Right:** If a model's shape shrinks when moving from Left to Right, it means the model loses intelligence when text is written in Singlish.

In [ ]:
# Display: Figure 8: Radar Chart — Multi-Task Performance
from IPython.display import Image, display
display(Image(r"figures/fig08_radar_chart.png", width=1100))

---


## 🏆 Figure 9: Script Robustness Ranking (Who is Most Consistent?)

### 📌 Simple Summary:
Which model is the most **script-robust**? That is, which model gives consistent results regardless of whether you write in native Sinhala letters or English letters?

### 🔍 How to Read This Chart:
- **Left Graph (Average Signed Gap):** Positive = favors native Sinhala; Negative = favors Singlish.
- **Right Graph (Mean Absolute Gap):** **LOWER IS BETTER.** A score close to 0.0 means the model treats both scripts with equal fairness.

### 💡 What the Data Tells Us:
1. **Most Robust Models:** `SmolLM3-3B` and `Zephyr-7B` have the lowest average gap (~1.5 pp). They perform with steady reliability across both scripts.
2. **Most Script-Biased Model:** `Qwen-3.5-9B` has a high gap (+11.6 pp). It is much smarter in native Sinhala than in Singlish.
3. **Worst Broken Model:** `LaMini-GPT-1.5B` has a huge gap because it crashes on native Sinhala.

In [ ]:
# Display: Figure 9: Script Robustness Score
from IPython.display import Image, display
display(Image(r"figures/fig09_robustness_score.png", width=1100))

---


## 🗺️ Figure 10: Complete Accuracy Grid (All Models × All Tasks)

### 📌 Simple Summary:
A master table showing the exact accuracy percentage for every model on every task in both scripts.

### 🔍 How to Read the Numbers:
- **Left Table:** Native Sinhala (`Unicode`).
- **Right Table:** Romanized Singlish (`Romanized`).
- **Darker Blue / Purple = Higher Accuracy (Better).**

### 💡 What the Data Tells Us:
1. **SOLD is the easiest task (accuracies up to 65.8%):** Detecting hate speech is easier than solving university-level exam questions.
2. **MMLU is the hardest task (accuracies range from 0.9% to 44.1%):** Answering complex academic questions in Sinhala requires deep language comprehension.

In [ ]:
# Display: Figure 10: Full Accuracy Heatmap (All Models x All Tasks)
from IPython.display import Image, display
display(Image(r"figures/fig10_full_accuracy_heatmap.png", width=1100))

---


## 📊 Figure 11: All Models Ranked Side-by-Side

### 📌 Simple Summary:
This chart lines up all 10 models side-by-side within each task so you can directly compare which model wins in each domain.

### 🔍 How to Read This Chart:
- **Top Plot:** Native Sinhala script.
- **Bottom Plot:** Romanized Singlish.
- Each color represents a specific AI model.

### 💡 What the Data Tells Us:
1. **MMLU Winner:** `Qwen-3.5-9B` is the clear champion on Sinhala MMLU (44.1%).
2. **SOLD Winner:** `Llama-3.1-8B` takes first place on offensive speech detection (65.8%).
3. **PIQA Winner:** `Qwen-3.5-4B` and `Qwen2-7B` lead on physical common sense reasoning (57.0%).

In [ ]:
# Display: Figure 11: Comprehensive Task Accuracy (All Models Side-by-Side)
from IPython.display import Image, display
display(Image(r"figures/fig11_comprehensive_task_accuracy.png", width=1100))

---


## ⚖️ Figure 12: Prediction Bias (Clean vs Offensive Balance)

### 📌 Simple Summary:
Do models have an unfair bias toward calling everything clean or calling everything offensive?

### 🔍 How to Read This Chart:
- **Dark Navy Bar (Gold):** The true real-world count of comments in the dataset.
- **Blue Bar:** What the model predicted in Native Sinhala.
- **Pink Bar:** What the model predicted in Romanized Singlish.
- **Ideal outcome:** The Blue and Pink bars should closely match the Dark Navy bar.

### 💡 What the Data Tells Us:
1. High-performing models (`Llama-3.1-8B`, `Qwen2-7B`) make predictions that match the real dataset distribution.
2. Small models suffer from extreme bias: they predict 'NOT' almost 100% of the time because they lack the vocabulary to detect offensive terms.

In [ ]:
# Display: Figure 12: SOLD Class Distribution per Model
from IPython.display import Image, display
display(Image(r"figures/fig12_sold_class_distribution.png", width=1100))

---


## 🎈 Figure 13: Accuracy vs Invalid Error Rate Bubble Plot

### 📌 Simple Summary:
Why do models get low scores? Is it because they picked the wrong answer, or because they failed to produce a valid response at all?

### 🔍 How to Read This Chart:
- **X-axis (Invalid Rate %):** Percentage of broken/unusable responses. **(Left is best, 0%)**.
- **Y-axis (Accuracy %):** Percentage of correct answers. **(Top is best, 100%)**.
- **Bubble Size:** Represents the number of test questions (MMLU = big bubble, PIQA = small bubble).
- **Desired Zone:** **Top-Left Corner** (High accuracy, zero invalid responses).

### 💡 What the Data Tells Us:
1. Reliable models cluster tightly in the top-left corner at 0% invalid errors.
2. Models in the bottom-right corner (like `LaMini` and `TinyLlama` on native Sinhala) score near zero purely because their output is 90%+ unparseable garbage.

In [ ]:
# Display: Figure 13: Bubble Chart — Accuracy vs Invalid Rate
from IPython.display import Image, display
display(Image(r"figures/fig13_bubble_accuracy_invalid.png", width=1100))

---


## 🔄 Figure 14: The Script Switching Impact (Romanized − Unicode)

### 📌 Simple Summary:
What happens when a user types in Singlish instead of native Sinhala? This chart shows the exact gain or drop in score.

### 🔍 How to Read This Chart:
- 🔵 **Blue Bar (Negative Value):** Model scored **higher in Native Sinhala**.
- 🌸 **Pink Bar (Positive Value):** Model scored **higher in Romanized Singlish**.
- **The Center Line (0.0):** Zero difference (Perfect script independence).

### 💡 What the Data Tells Us:
1. On **Sinhala MMLU**, switching to Singlish causes a huge drop (-15.5 pp) for advanced models like `Qwen-3.5-9B`.
2. On **SOLD**, the difference is very small (bars stay close to 0.0), showing that offensive keywords are recognizable in both scripts.

In [ ]:
# Display: Figure 14: Delta Plot — Script Switch (Romanized - Unicode)
from IPython.display import Image, display
display(Image(r"figures/fig14_delta_plot.png", width=1100))

---


## 🎛️ Figure 15: Executive Project Summary Dashboard

### 📌 Simple Summary:
A high-level executive dashboard summarizing the entire research project in one clean visual view.

### 🔍 Key Highlights on the Dashboard:
- **Best Knowledge Model (MMLU):** `Qwen-3.5-9B` (44.1% Accuracy)
- **Best Safety & Moderation Model (SOLD):** `Llama-3.1-8B-Instruct` (65.8% Accuracy)
- **Most Script-Robust Model:** `SmolLM3-3B` & `Zephyr-7B`
- **Average Script Gap Across All Tests:** +2.4 pp favoring native Sinhala script.

In [ ]:
# Display: Figure 15: Summary Dashboard
from IPython.display import Image, display
display(Image(r"figures/fig15_dashboard.png", width=1100))

---


## 🌍 Figure 16: Global PIQA — Physical Commonsense Reasoning in Sinhala

### 📌 Simple Summary:
Physical Intuition QA tests common sense knowledge about the physical world (e.g., 'To separate egg yolks, you should use a plastic bottle'). Here, questions are translated into Sinhala.

### 🔍 How to Read This Chart:
- **Blue Bar = Native Sinhala** | **Pink Bar = Romanized Singlish**
- **Dashed Line at 50%:** Random guessing baseline (since there are 2 choices).

### 💡 What the Data Tells Us:
1. Most models hover around **50% - 57%**, showing that physical commonsense reasoning in Sinhala remains challenging for current AI.
2. Models like `Qwen-3.5-4B` and `Llama-3.1-8B` lead the pack with over 53% accuracy in both scripts.

In [ ]:
# Display: Figure 16: Global PIQA Comparison
from IPython.display import Image, display
display(Image(r"figures/fig16_piqa_comparison.png", width=1100))

---


## 🎓 Final Conclusions & Recommendations

### 1. Key Takeaways

| Area | Finding |
|:---|:---|
| **Overall Best Model** | **Llama-3.1-8B-Instruct** and **Qwen-3.5-9B** are the strongest models for Sinhala NLP applications. |
| **Best Script for AI** | **Native Unicode Sinhala** produces significantly better results on knowledge-heavy tasks (MMLU). |
| **Singlish Performance** | Singlish works decently well on simple tasks (hate speech detection), but complex reasoning drops by up to 15%. |
| **Small Model Limitation** | Models under 3B parameters (`TinyLlama`, `LaMini`) are not suitable for Sinhala script without dedicated fine-tuning. |

### 2. Practical Recommendations for Sinhala NLP Developers:
- If you are building a **Sinhala Chatbot or Q&A System**, always convert user input into **Native Unicode Sinhala** before feeding it into the LLM for highest accuracy.
- If you are building a **Social Media Content Moderation Tool**, `Llama-3.1-8B` can effectively handle both native Sinhala script and Singlish.

---
**Sinhala Script Robustness Project**
